# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 03 — Logistic Regression

---

### Purpose
Train and evaluate a Logistic Regression classifier on all four feature sets
produced in Notebook 01. Produce comprehensive evaluation artefacts including
confusion matrices, classification reports, and ROC curves.

### Objectives
1. Load all feature matrices
2. Train Logistic Regression on each feature set
3. Evaluate with accuracy, precision, recall, F1 (macro + weighted), ROC-AUC
4. Generate confusion matrices and ROC curves
5. Inspect model coefficients (top discriminative features)
6. Save trained models
7. Show prediction examples

### Notebook Outline
1. Imports
2. Configuration
3. Load TF-IDF Features
4. Load Character N-Gram Features
5. Load Stylometric Features
6. Load Embedding Features
7. Training
8. Evaluation
9. Confusion Matrix
10. Classification Report
11. ROC Curves
12. Feature Coefficients
13. Model Saving
14. Prediction Examples
15. Notebook Summary

---

## 1. Imports

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.figure_factory as ff

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.classifiers import LogisticRegressionClassifier
from src.feature_engineering.utils import load_feature_matrix, setup_logger
from src.evaluation.evaluator import ModelEvaluator
from src.visualization.plots import (
    plot_confusion_matrix, plot_roc_curves, plot_feature_importance
)
from src.utils.helpers import (
    set_global_seed, train_test_val_split, load_yaml, make_output_dirs,
    display_metrics_table, print_section_header,
)

print('✅ All libraries imported successfully.')

---

## 2. Configuration

In [ ]:
# ── Load training configuration ────────────────────────────────────────────────
cfg = load_yaml(PROJECT_ROOT / 'configs' / 'training.yaml')

RANDOM_SEED  = cfg['random_seed']
TEST_SIZE    = cfg['evaluation']['test_size']
VAL_SIZE     = cfg['evaluation']['val_size']
LR_CFG       = cfg['logistic_regression']

set_global_seed(RANDOM_SEED)

# ── Output directories ─────────────────────────────────────────────────────────
DIR_MODELS  = PROJECT_ROOT / cfg['output']['models_dir']
DIR_FIGURES = PROJECT_ROOT / cfg['output']['figures_dir']
DIR_OUTPUTS = PROJECT_ROOT / cfg['output']['outputs_dir']
make_output_dirs(DIR_MODELS, DIR_FIGURES, DIR_OUTPUTS)

# ── Feature paths ──────────────────────────────────────────────────────────────
FEAT_CFG = cfg['features']

setup_logger(str(PROJECT_ROOT / cfg['logging']['log_file']), cfg['logging']['level'])
logger = logging.getLogger(__name__)

print(f'Random Seed : {RANDOM_SEED}')
print(f'Test Size   : {TEST_SIZE}')
print(f'LR Config   : {LR_CFG}')

---

## 3. Load TF-IDF Features

In [ ]:
# ── Load TF-IDF word feature matrix — Pipeline A ──────────────────────────────
TFIDF_MATRIX_PATH = PROJECT_ROOT / FEAT_CFG['tfidf']['fingerprint']
TFIDF_LABELS_PATH = PROJECT_ROOT / FEAT_CFG['labels']['fingerprint']

X_tfidf, y_tfidf = load_feature_matrix(TFIDF_MATRIX_PATH, TFIDF_LABELS_PATH)
classes = np.load(str(TFIDF_MATRIX_PATH).replace('.npz', '').replace(
    'tfidf_fingerprint', 'classes_tfidf_fingerprint') + '.npy', allow_pickle=True)

print(f'TF-IDF Matrix : {X_tfidf.shape}')
print(f'Labels        : {y_tfidf.shape}  classes={list(classes)}')

In [ ]:
# ── Train/Test split — TF-IDF ─────────────────────────────────────────────────
X_tr_tf, X_val_tf, X_te_tf, y_tr_tf, y_val_tf, y_te_tf = train_test_val_split(
    X_tfidf, y_tfidf,
    test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)
print(f'Train: {X_tr_tf.shape[0]}  Val: {X_val_tf.shape[0]}  Test: {X_te_tf.shape[0]}')

---

## 4. Load Character N-Gram Features

In [ ]:
# ── Load Char N-Gram features — Pipeline A ────────────────────────────────────
CHAR_MATRIX_PATH = PROJECT_ROOT / FEAT_CFG['char']['fingerprint']
CHAR_LABELS_PATH = PROJECT_ROOT / FEAT_CFG['labels']['fingerprint']

X_char, y_char = load_feature_matrix(CHAR_MATRIX_PATH, CHAR_LABELS_PATH)

X_tr_ch, X_val_ch, X_te_ch, y_tr_ch, y_val_ch, y_te_ch = train_test_val_split(
    X_char, y_char, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)
print(f'Char Matrix : {X_char.shape}  → Train {X_tr_ch.shape[0]}  Test {X_te_ch.shape[0]}')

---

## 5. Load Stylometric Features

In [ ]:
# ── Load Stylometric features — Pipeline A ────────────────────────────────────
STYLE_MATRIX_PATH = PROJECT_ROOT / FEAT_CFG['style']['fingerprint']
STYLE_LABELS_PATH = PROJECT_ROOT / FEAT_CFG['labels']['fingerprint']

X_style, y_style = load_feature_matrix(STYLE_MATRIX_PATH, STYLE_LABELS_PATH)

# Stylometric is dense — convert to numpy array if stored as npz
if sp.issparse(X_style):
    X_style = X_style.toarray()

X_tr_st, X_val_st, X_te_st, y_tr_st, y_val_st, y_te_st = train_test_val_split(
    X_style, y_style, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)
print(f'Style Matrix : {X_style.shape}  → Train {X_tr_st.shape[0]}  Test {X_te_st.shape[0]}')

---

## 6. Load Embedding Features

In [ ]:
# ── Load Embedding features — Pipeline A ─────────────────────────────────────
EMB_DIR = PROJECT_ROOT / 'data' / 'features' / 'embedding'

emb_data = np.load(str(EMB_DIR / 'emb_fingerprint.npz'))
X_emb    = emb_data['embeddings']
y_emb    = np.load(str(EMB_DIR / 'labels_emb_fingerprint.npy'))

X_tr_em, X_val_em, X_te_em, y_tr_em, y_val_em, y_te_em = train_test_val_split(
    X_emb, y_emb, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)
print(f'Embedding Matrix : {X_emb.shape}  → Train {X_tr_em.shape[0]}  Test {X_te_em.shape[0]}')

---

## 7. Training

In [ ]:
# ── Helper: train + evaluate one configuration ─────────────────────────────────
def run_logistic_regression(X_train, X_test, y_train, y_test, feature_set: str):
    """Train a Logistic Regression classifier and return a ModelEvaluator."""
    model = LogisticRegressionClassifier(cfg=LR_CFG)
    model.fit(X_train, y_train, feature_set=feature_set)

    evaluator = ModelEvaluator(
        model_name='logistic_regression',
        feature_set=feature_set,
        classes=classes,
    )
    evaluator.evaluate(
        estimator=model.model,
        X_test=X_test,
        y_test=y_test,
        train_time=model.train_time_,
    )
    return model, evaluator

In [ ]:
print_section_header('Training Logistic Regression — TF-IDF Features')
lr_tfidf, eval_tfidf = run_logistic_regression(
    X_tr_tf, X_te_tf, y_tr_tf, y_te_tf, 'tfidf'
)

print_section_header('Training Logistic Regression — Char N-Gram Features')
lr_char, eval_char = run_logistic_regression(
    X_tr_ch, X_te_ch, y_tr_ch, y_te_ch, 'char'
)

print_section_header('Training Logistic Regression — Stylometric Features')
lr_style, eval_style = run_logistic_regression(
    X_tr_st, X_te_st, y_tr_st, y_te_st, 'style'
)

print_section_header('Training Logistic Regression — Embedding Features')
lr_emb, eval_emb = run_logistic_regression(
    X_tr_em, X_te_em, y_tr_em, y_te_em, 'embedding'
)

print('\n✅ All Logistic Regression variants trained.')

---

## 8. Evaluation

In [ ]:
# ── Metrics summary table across all feature sets ─────────────────────────────
results = pd.DataFrame([
    eval_tfidf.to_series(),
    eval_char.to_series(),
    eval_style.to_series(),
    eval_emb.to_series(),
])

display_cols = [
    'model_name', 'feature_set', 'accuracy',
    'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted',
    'train_time_s', 'pred_time_s',
]
results[display_cols].style.highlight_max(
    subset=['accuracy', 'f1_macro', 'f1_weighted'],
    color='lightgreen'
).format(precision=4)

---

## 9. Confusion Matrix

In [ ]:
# ── Confusion matrices for each feature set ───────────────────────────────────
import numpy as np

for evaluator, label in [
    (eval_tfidf, 'TF-IDF'),
    (eval_char,  'Char N-Gram'),
    (eval_style, 'Stylometric'),
    (eval_emb,   'Embedding'),
]:
    cm = np.array(evaluator.results_['confusion_matrix'])
    out_path = DIR_FIGURES / f'lr_cm_{evaluator.feature_set}.png'
    plot_confusion_matrix(
        cm=cm,
        class_names=list(classes),
        title=f'Logistic Regression — {label} — Confusion Matrix',
        out_path=out_path,
        normalize=True,
    )
    print(f'✅ Confusion matrix saved: {out_path.name}')

---

## 10. Classification Report

In [ ]:
# ── Print and save classification report for best feature set ─────────────────
best_eval = max(
    [eval_tfidf, eval_char, eval_style, eval_emb],
    key=lambda e: e.results_['f1_macro']
)

print(f'Best feature set: {best_eval.feature_set}')
print(f"Macro F1 : {best_eval.results_['f1_macro']:.4f}")
print()
print(best_eval.results_['classification_report'])

# Save report to outputs/
best_eval.save_classification_report(
    DIR_OUTPUTS / f'lr_{best_eval.feature_set}_classification_report.txt'
)

---

## 11. ROC Curves

In [ ]:
# ── ROC curves for the best-performing feature set ────────────────────────────
if best_eval.results_.get('y_proba') is not None:
    roc_path = DIR_FIGURES / f'lr_roc_{best_eval.feature_set}.png'
    plot_roc_curves(
        y_test=best_eval.results_['y_test'],
        y_proba=best_eval.results_['y_proba'],
        class_names=list(classes),
        title=f'Logistic Regression — {best_eval.feature_set} — ROC Curves',
        out_path=roc_path,
    )
    print(f'✅ ROC curves saved: {roc_path.name}')
else:
    print('ROC curves require predict_proba support.')

---

## 12. Feature Coefficients

In [ ]:
# ── Top discriminative TF-IDF features by coefficient magnitude ───────────────
# Logistic Regression coefficients are a (n_classes × n_features) matrix.
# The absolute mean across classes measures overall feature importance.

coef_matrix = lr_tfidf.model.coef_   # shape: (n_classes, n_features)
mean_abs_coef = np.abs(coef_matrix).mean(axis=0)

# Reconstruct feature names from saved vectorizer
from src.feature_engineering.tfidf_extractor import TFIDFExtractor
tfidf_extractor = TFIDFExtractor.load(PROJECT_ROOT / 'data' / 'features' / 'tfidf')
word_feature_names = list(tfidf_extractor.get_feature_names('word'))

coef_path = DIR_FIGURES / 'lr_tfidf_top_features.png'
plot_feature_importance(
    importances=mean_abs_coef,
    feature_names=word_feature_names,
    title='Logistic Regression — Top TF-IDF Features (Mean |Coefficient|)',
    out_path=coef_path,
    top_n=30,
)
print(f'✅ Feature importance chart saved: {coef_path.name}')

---

## 13. Model Saving

In [ ]:
# ── Save all Logistic Regression variants ─────────────────────────────────────
for model, suffix in [
    (lr_tfidf, 'tfidf'),
    (lr_char,  'char'),
    (lr_style, 'style'),
    (lr_emb,   'embedding'),
]:
    saved_path = model.save(DIR_MODELS / 'logistic_regression', suffix=suffix)
    print(f'✅ Saved: {saved_path.name}')

---

## 14. Prediction Examples

In [ ]:
# ── Show 10 prediction examples with true vs predicted labels ─────────────────
# Using the best-performing model
best_lr = {  # map feature_set → (model, X_test, y_test)
    'tfidf':     (lr_tfidf, X_te_tf, y_te_tf),
    'char':      (lr_char,  X_te_ch, y_te_ch),
    'style':     (lr_style, X_te_st, y_te_st),
    'embedding': (lr_emb,   X_te_em, y_te_em),
}[best_eval.feature_set]

model, X_te, y_te = best_lr
y_pred_sample = model.predict(X_te[:10])
y_proba_sample = model.predict_proba(X_te[:10])

example_df = pd.DataFrame({
    'True Label':      [classes[i] for i in y_te[:10]],
    'Predicted Label': [classes[i] for i in y_pred_sample],
    'Correct':         y_te[:10] == y_pred_sample,
    'Max Confidence':  np.max(y_proba_sample, axis=1).round(4),
})
example_df

---

## 15. Notebook Summary

### Logistic Regression Results Summary

| Feature Set | Macro F1 | Weighted F1 | Train Time |
|---|---|---|---|
| TF-IDF | *(run to populate)* | *(run to populate)* | *(run to populate)* |
| Char N-Gram | *(run to populate)* | *(run to populate)* | *(run to populate)* |
| Stylometric | *(run to populate)* | *(run to populate)* | *(run to populate)* |
| Embeddings | *(run to populate)* | *(run to populate)* | *(run to populate)* |

### Key Observations (to be completed after execution)
- Best feature set for Logistic Regression: **TBD**
- Most challenging LLM to classify: **TBD**
- Notable coefficient features: **TBD**

### Next Steps
→ **Notebook 04**: Linear SVM with identical evaluation structure

---
*Fingerprint Project — Logistic Regression — Complete*